# Lecture 2: Estimators and information
**Guided student notebook · about 90 minutes**

Estimate π from random points (15 min), study repeated estimates (25 min),
and explore parameter degeneracy and Fisher information (40 min), with
10 min for discussion. The sphere and correlated-noise extensions are optional
and sit outside the 90-minute session.

Run the cells in order. Replace `None` at each **TODO** and answer the short
interpretation questions. Supplied displays show reminders until the exercises
are complete. [Repetition and plotting helpers](lecture_02_helpers.py) are supplied;
you do not need to implement them.

Lecture references: [estimators and information summary](../handouts/lecture_02_summary.pdf),
[Monte Carlo π](../01_probability/monte_carlo_pi.ipynb), and
[parameter degeneracy](../02_estimators/parameter_degeneracy.ipynb).

## Setup
Reuse the environment from lecture one. In a terminal at the repository root:

```bash
cd tutorials
source .venv/bin/activate
uv pip install --python .venv -r requirements.txt
python -m jupyterlab
```

For a first installation, install [uv](https://docs.astral.sh/uv/getting-started/installation/)
and run `uv venv --python 3.12` after `cd tutorials`, before activating the environment.
In Windows PowerShell, replace the activation line with `.venv\Scripts\Activate.ps1`.
Select this environment as the notebook kernel and keep `lecture_02_helpers.py`
and `requirements.txt` alongside the notebooks. Restart the kernel after installing packages.

In [ ]:
import jax
import jax.numpy as jnp
from jax import random
import matplotlib.pyplot as plt

from lecture_02_helpers import (
    repeat_estimator, plot_circle, plot_sampling_distributions, plot_likelihoods,
)

jax.config.update("jax_enable_x64", True)
plt.style.use("default")
key = random.key(2026)

setup_sample = random.normal(random.key(101), shape=(32,))
print(f"JAX {jax.__version__}; sample shape: {setup_sample.shape}")
plt.hist(setup_sample, bins=8, density=True)
plt.xlabel("x")
plt.ylabel("Density")
plt.show()

## 1. Estimate π from random points · 15 min
Draw $n$ independent uniform points in the square $[-1,1]^2$.
Let $I_i$ be 1 if point $i$ lies inside the unit circle and 0 otherwise.
The fraction inside estimates the circle's area divided by the square's area.
Here $\pi$ is the geometric constant, a known target for checking our procedure.

**Task.** Complete the sampling, inclusion condition and estimate below.
Use `random.uniform` with `shape=(n, 2)`, `minval=-1.0`, `maxval=1.0`.
For each row, sum the two squared coordinates using `axis=1` and compare with 1.
The estimate is $\widehat\pi=4\bar I$; `jnp.mean` also works on a Boolean mask.

In [ ]:
def estimate_pi(draw_key, n):
    points = None    # TODO: n uniform points in the square.
    inside = None    # TODO: a Boolean mask with shape (n,).
    estimate = None  # TODO: four times the fraction inside.
    return points, inside, estimate

In [ ]:
key, draw_key = random.split(key)
points, inside, pi_estimate = estimate_pi(draw_key, 1000)
if any(value is None for value in (points, inside, pi_estimate)):
    print("Complete estimate_pi, then rerun this cell.")
else:
    print(f"Estimate: {float(pi_estimate):.4f}; true π: {float(jnp.pi):.4f}")
    plot_circle(points, inside)

**Explain.** Why multiply the fraction inside by four? What is the difference
between the estimator and this numerical estimate? Rerun the cell with a fresh
key: must the next estimate be closer to π?

**Your answer:**

## 2. Repeat the experiment · 25 min
Generate $R$ independent experiments, each containing $n$ points. The resulting
$R$ estimates describe the **sampling distribution** of the estimator at that $n$.
The supplied `repeat_estimator` uses separate JAX keys for the experiments.

**Task.** Summarise the repeated estimates $t_1,\ldots,t_R$:

$$\widehat b=\bar t-\pi,\qquad
\widehat v=\frac1R\sum_r(t_r-\bar t)^2,\qquad
\widehat{\mathrm{MSE}}=\frac1R\sum_r(t_r-\pi)^2.$$

Use `jnp.mean` and `jnp.var(..., ddof=0)`. We use the empirical-distribution
variance (denominator $R$) here, so the finite-array identity
$\widehat{\mathrm{MSE}}=\widehat v+\widehat b^2$ holds exactly apart from rounding.
Lecture one's `ddof=1` instead estimates a population variance without bias.

Also complete the theoretical variance function. Start from independent
$I_i\sim\mathrm{Bernoulli}(p)$ with $p=\pi/4$ and
$\mathrm{Var}(I_i)=p(1-p)$. Use
$\mathrm{Var}(4\bar I)=16p(1-p)/n$ and simplify.

In [ ]:
def estimator_summary(estimates, truth):
    bias = None      # TODO: mean estimate minus truth.
    variance = None  # TODO: empirical variance, ddof=0.
    mse = None       # TODO: mean squared error relative to truth.
    return bias, variance, mse

def pi_variance(n):
    return None  # TODO: the exact sampling variance as a function of n.

**Predict first.** How should the variance and standard error change when
the number of points per estimate increases from 100 to 400? What changes
if you increase the number of repetitions instead?

In [ ]:
sample_sizes = (100, 400, 1600)
repetitions = 2000  # Independent estimates at each n, not points per estimate.
pi_samples = {}
pi_summaries = {}

if pi_estimate is None:
    print("Complete estimate_pi and rerun its display cell first.")
else:
    for n in sample_sizes:
        key, draw_key = random.split(key)
        pi_samples[n] = repeat_estimator(draw_key, n, repetitions, estimate_pi)

    print(f"R = {repetitions} independent experiments at each n")
    print(f"{'n':>6} {'Bias':>10} {'Variance':>10} {'MSE':>10} {'Theory var':>12}")
    for n, estimates in pi_samples.items():
        summary = estimator_summary(estimates, jnp.pi)
        theory = pi_variance(n)
        if any(value is None for value in (*summary, theory)):
            print("Complete estimator_summary and pi_variance, then rerun this cell.")
            break
        pi_summaries[n] = summary
        bias, variance, mse = summary
        print(f"{n:6d} {float(bias):10.5f} {float(variance):10.5f} "
              f"{float(mse):10.5f} {float(theory):12.5f}")

    plot_sampling_distributions(
        {f"n = {n}": pi_samples[n] for n in (100, 400)}, jnp.pi, "Estimate of π",
    )

**Explain.**

1. Derive $\mathbb E[\widehat\pi]$ and $\mathrm{Var}(\widehat\pi)$ from the indicator model.
2. Does a nonzero empirical bias prove that the estimator is biased?
3. Compare the variance reduction with your prediction. Why is the estimator consistent?
4. Distinguish increasing $n$ from increasing $R$. Would a running estimate from
   one growing set of points provide independent repetitions?

**Your answer:**

## 3. Degeneracy and Fisher information · 40 min
One experiment produces the two-component observation
$$x_1=\theta+\phi+\epsilon_1,\qquad x_2=\phi+\epsilon_2,
\qquad \epsilon_1,\epsilon_2\overset{\mathrm{iid}}{\sim}\mathcal N(0,\sigma^2).$$
The noise standard deviation $\sigma$ is known. We want to estimate $\theta$;
$\phi$ is a **nuisance parameter** whose contribution overlaps with that of $\theta$.

Start with the illustrative observation $(x_1,x_2)=(10,3)$.
The supplied plots show relative likelihood $L/L_{\max}$ at levels
0.1, 0.4, 0.7 and 0.9. They are not normalised probability densities over parameters.

In [ ]:
theta_true, phi_true, sigma = 7.0, 3.0, 1.0
observation = jnp.array([10.0, 3.0])
plot_likelihoods(observation, sigma)

**Explain.** Which combination of $\theta$ and $\phi$ does $x_1$ constrain?
Give two parameter pairs with the same maximum likelihood from $x_1$ alone.
Why does adding $x_2$ close the contours?

**Your answer:**

Compare two situations using the **same observations**: $\phi$ is supplied
at its true value, or $\phi$ must be fitted jointly with $\theta$.

**Task.** Complete both estimators. With $\phi$ known, remove its contribution
from $x_1$. With $\phi$ fitted, first use $\widehat\phi=x_2$.
Each row of `data` is one experiment; use `data[:, 0]` and `data[:, 1]`.

In [ ]:
def theta_estimators(data, phi_known):
    known = None   # TODO: theta estimates with phi known exactly.
    fitted = None  # TODO: theta estimates with phi fitted from x2.
    return known, fitted

The simulation below keeps both true parameters fixed across 10,000 independent
experiments. Each experiment still contains just **one two-component observation**.
We do not pool these repetitions into a larger fit.

In [ ]:
fisher_repetitions = 10_000
key, draw_key = random.split(key)
mean_data = jnp.array([theta_true + phi_true, phi_true])
data = mean_data + sigma * random.normal(draw_key, shape=(fisher_repetitions, 2))
theta_known, theta_fitted = theta_estimators(data, phi_true)

**Task.** Calculate the sample variance of each array of estimates using `ddof=1`.
Predict which is larger, then inspect the supplied comparison.

In [ ]:
known_variance = None   # TODO: sample variance of theta_known.
fitted_variance = None  # TODO: sample variance of theta_fitted.

In [ ]:
if any(value is None for value in (theta_known, theta_fitted, known_variance, fitted_variance)):
    print("Complete both estimators and variances, then rerun the simulation and comparison.")
else:
    print(f"True theta: {theta_true}; repetitions: {fisher_repetitions}")
    print(f"{'Treatment of phi':<18} {'Mean':>10} {'Variance':>12}")
    for label, estimates, variance in (
        ("Known", theta_known, known_variance), ("Fitted", theta_fitted, fitted_variance),
    ):
        print(f"{label:<18} {float(jnp.mean(estimates)):10.4f} {float(variance):12.4f}")
    plot_sampling_distributions(
        {"phi known": theta_known, "phi fitted": theta_fitted}, theta_true, "Estimate of theta",
    )

For the parameter order $(\theta,\phi)$, write the mean as
$A(\theta,\phi)^{\mathsf T}$, where
$$A=\begin{pmatrix}1&1\\0&1\end{pmatrix},\qquad C=\sigma^2 I_2.$$
Since the covariance is known and independent of the parameters, the Fisher
information for **one experiment** is $F=A^{\mathsf T}C^{-1}A$.

**Task.** Compute $F$ and its inverse with `jnp.linalg.inv` and matrix multiplication `@`.
Then obtain the variance bounds $1/F_{\theta\theta}$ for known $\phi$ and
$(F^{-1})_{\theta\theta}$ when both parameters are fitted. The theta entry is `[0, 0]`.
These are Cramér–Rao bounds for unbiased estimators under the model's regularity
conditions; our linear Gaussian estimators attain them exactly.

In [ ]:
A = jnp.array([[1.0, 1.0], [0.0, 1.0]])
C = sigma**2 * jnp.eye(2)
F = None                    # TODO: Fisher matrix for one experiment.
F_inverse = None            # TODO: matrix inverse of F.
variance_known_bound = None   # TODO: reciprocal of the theta diagonal entry of F.
variance_fitted_bound = None  # TODO: theta diagonal entry of F_inverse.

In [ ]:
if any(value is None for value in (
    F, F_inverse, variance_known_bound, variance_fitted_bound, known_variance, fitted_variance,
)):
    print("Complete the variance and Fisher calculations, then rerun this cell.")
else:
    print("Fisher matrix:\n", F)
    print("Inverse Fisher matrix:\n", F_inverse)
    print(f"{'Treatment of phi':<18} {'Empirical var':>14} {'Fisher bound':>14}")
    print(f"{'Known':<18} {float(known_variance):14.4f} {float(variance_known_bound):14.4f}")
    print(f"{'Fitted':<18} {float(fitted_variance):14.4f} {float(variance_fitted_bound):14.4f}")

**Explain.**

1. Write the estimation error for each estimator in terms of $\epsilon_1,\epsilon_2$.
   Derive its expectation and variance. Does fitting $\phi$ double the standard deviation?
2. Why are $1/F_{\theta\theta}$ and $(F^{-1})_{\theta\theta}$ different?
3. Should you multiply $F$ by the number of repetitions for the comparison above?

**Your answer:**

## Discussion · 10 min
Distinguish observations within one estimate from independent repetitions used
to assess an estimator. Why can fitting a nuisance parameter reduce precision
even when both estimators are unbiased? Would the same loss occur with independent
noise if the mean model instead separated as $x_1=\theta+\epsilon_1$,
$x_2=\phi+\epsilon_2$?

**Your answer:**

## Optional A. A sphere inside a cube
**Outside the core 90 minutes.** Adapt the first estimator to uniform points in
$[-1,1]^3$ and the unit sphere. The sphere has volume $4\pi/3$ and the cube has
volume 8. Derive the factor converting the hit fraction into an estimate of $\pi$.

**Task.** Complete the three lines and compare the two estimators at the same $n$.
Use `shape=(n, 3)` and again sum squared coordinates along `axis=1`.

In [ ]:
def estimate_pi_3d(draw_key, n):
    points = None    # TODO: uniform points in the cube.
    inside = None    # TODO: membership of the unit sphere.
    estimate = None  # TODO: multiply the hit fraction by the geometric factor.
    return points, inside, estimate

In [ ]:
extension_n = 400
key, check_key = random.split(key)
sphere_points, sphere_inside, sphere_estimate = estimate_pi_3d(check_key, extension_n)
sphere_samples = {}
if pi_estimate is None or any(value is None for value in (sphere_points, sphere_inside, sphere_estimate)):
    print("Complete both geometric estimators before running this optional comparison.")
else:
    print(f"n = {extension_n} points per estimate; R = {repetitions} independent experiments")
    print(f"{'Geometry':<12} {'Sample variance':>18} {'Exact variance':>18}")
    for label, estimator, factor in (("Circle", estimate_pi, 4), ("Sphere", estimate_pi_3d, 6)):
        key, draw_key = random.split(key)
        estimates = repeat_estimator(draw_key, extension_n, repetitions, estimator)
        sphere_samples[label] = estimates
        exact_variance = jnp.pi * (factor - jnp.pi) / extension_n
        print(f"{label:<12} {float(jnp.var(estimates, ddof=1)):18.5f} {float(exact_variance):18.5f}")
    plot_sampling_distributions(sphere_samples, jnp.pi, "Estimate of π")

**Explain.** Derive the hit probability and variance of the sphere estimator.
Why is its variance larger at the same number of points? Does the standard-error
scaling with $n$ change?

**Your answer:**

## Optional B. Correlated measurement noise
**Outside the core 90 minutes.** Keep the same mean model but replace independent
noise by the known covariance
$$C_\rho=\sigma^2\begin{pmatrix}1&\rho\\\rho&1\end{pmatrix},\qquad -1<\rho<1.$$
We still fit only $\theta$ and, when unknown, $\phi$. Correlation of the measurement
noise is distinct from correlation between the fitted parameters.

The efficient estimator for known $\phi$, using both measurements, is supplied:
$$\widehat\theta_{\mathrm{known}}=x_1-\phi-\rho(x_2-\phi),\qquad
\mathrm{Var}(\widehat\theta_{\mathrm{known}})=\sigma^2(1-\rho^2).$$
The joint-fit estimator remains $x_1-x_2$, with variance $2\sigma^2(1-\rho)$.
Keeping the old known-$\phi$ estimator would ignore useful information in $x_2$.

**Predict first.** How does the variance of $x_1-x_2$ change for positive and
negative $\rho$? Why can knowing $\phi$ now let us use $x_2$ to correct some noise?

**Task.** Adapt the supplied simulation: draw zero-mean multivariate Gaussian
noise with `random.multivariate_normal(..., shape=(repetitions,))`; the resulting
array has shape `(repetitions, 2)`. Then recompute $F=A^{\mathsf T}C_\rho^{-1}A$.
Keep $\rho$ strictly between $-1$ and $1$ so the covariance is invertible.

In [ ]:
def correlated_noise_and_fisher(draw_key, rho, repetitions):
    C_rho = sigma**2 * jnp.array([[1.0, rho], [rho, 1.0]])
    noise = None    # TODO: random.multivariate_normal with mean jnp.zeros(2).
    F_rho = None    # TODO: Fisher matrix using C_rho.
    return noise, F_rho

In [ ]:
correlation_results = {}
for rho in (-0.6, 0.0, 0.6):
    key, draw_key = random.split(key)
    noise, F_rho = correlated_noise_and_fisher(draw_key, rho, fisher_repetitions)
    if noise is None or F_rho is None:
        print("Complete the optional correlated-noise simulation and Fisher calculation.")
        break
    correlated_data = mean_data + noise
    x1, x2 = correlated_data[:, 0], correlated_data[:, 1]
    known = x1 - phi_true - rho * (x2 - phi_true)  # Supplied optimal correction.
    fitted = x1 - x2
    bounds = (1 / F_rho[0, 0], jnp.linalg.inv(F_rho)[0, 0])
    exact = (sigma**2 * (1 - rho**2), 2 * sigma**2 * (1 - rho))
    correlation_results[rho] = (known, fitted, F_rho)

    print(f"Noise correlation rho = {rho:+.1f}")
    print(f"{'phi':<10} {'Sample var':>12} {'Exact var':>12} {'Fisher bound':>14}")
    for label, estimates, variance, bound in zip(("Known", "Fitted"), (known, fitted), exact, bounds):
        print(f"{label:<10} {float(jnp.var(estimates, ddof=1)):12.4f} "
              f"{float(variance):12.4f} {float(bound):14.4f}")
    plot_likelihoods(observation, sigma, rho=rho)
    plot_sampling_distributions(
        {"phi known": known, "phi fitted": fitted}, theta_true, "Estimate of theta",
    )

**Explain.** Why does $\rho(x_2-\phi)$ predict part of the first measurement's noise?
How do the joint likelihood and repeated-estimate widths change with $\rho$?
Does fitting $\phi$ ever beat knowing its true value when both estimators use
both measurements optimally? Do the noise correlation and parameter correlation
have to have the same sign?

**Your answer:**